# Lab 6: CNN on MNIST with Custom Mini-Batch Gradient Descent

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import time

# Load and prepare 10,000 points from MNIST
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()
num_points = 10000

x_train = x_train[:num_points].reshape(-1, 28, 28, 1) / 255.0
y_train = y_train[:num_points]
x_test = x_test[:2000].reshape(-1, 28, 28, 1) / 255.0
y_test = y_test[:2000]

print(f"Training data shape: {x_train.shape}")
print(f"Test data shape: {x_test.shape}")

Training data shape: (10000, 28, 28, 1)
Test data shape: (2000, 28, 28, 1)


In [2]:
# Build the CNN model using Keras
model = models.Sequential([
    layers.Conv2D(4, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(10, activation='softmax')
])

model.summary()

C:\Users\saira\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 4)      │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 4)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 676)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │         6,770 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,810 (26.60 KB)

 Trainable params: 6,810 (26.60 KB)

 Non-trainable params: 0 (0.00 B)

## Custom Mini-Batch Gradient Descent Loop
Clean, standard output processing all 10,000 images.

In [3]:
epochs = 100
batch_size = 32
learning_rate = 0.01

optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

# Using ceil ensures we process ALL 10,000 images (313 batches)
num_batches = int(np.ceil(len(x_train) / batch_size))
print("Training with Custom Mini-Batch Gradient Descent...")
start_time = time.time()

for epoch in range(epochs):
    indices = np.random.permutation(len(x_train))
    x_shuf = x_train[indices]
    y_shuf = y_train[indices]
    
    epoch_loss = 0
    correct = 0
    
    for b in range(num_batches):
        X_batch = x_shuf[b*batch_size : (b+1)*batch_size]
        Y_batch = y_shuf[b*batch_size : (b+1)*batch_size]
        current_batch_size = len(X_batch)
        
        with tf.GradientTape() as tape:
            preds = model(X_batch, training=True)
            loss = loss_fn(Y_batch, preds)
            
        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        
        # We multiply by current_batch_size to weight the loss correctly for the final batch
        epoch_loss += loss.numpy() * current_batch_size
        correct += np.sum(np.argmax(preds.numpy(), axis=1) == Y_batch)
        
    avg_loss = epoch_loss / len(x_train)
    accuracy = correct / len(x_train)
    
    print(f"Epoch {epoch+1:03d}/{epochs} | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.4f}")
    
print(f"\nTotal time taken: {time.time() - start_time:.2f}s")

Training with Custom Mini-Batch Gradient Descent...


Epoch 001/100 | Loss: 1.5402 | Accuracy: 0.5734


Epoch 002/100 | Loss: 0.5139 | Accuracy: 0.8598


Epoch 003/100 | Loss: 0.3974 | Accuracy: 0.8870


Epoch 004/100 | Loss: 0.3594 | Accuracy: 0.8982


...

Epoch 096/100 | Loss: 0.0982 | Accuracy: 0.9736


Epoch 097/100 | Loss: 0.0974 | Accuracy: 0.9733


Epoch 098/100 | Loss: 0.0953 | Accuracy: 0.9740


Epoch 099/100 | Loss: 0.0941 | Accuracy: 0.9741


Epoch 100/100 | Loss: 0.0927 | Accuracy: 0.9746

Total time taken: 526.96s
